# Binding Pocket Detection

In [1]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path
import pickle

import numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import sciapi
import scifile
import scishow
import caddpy

In [2]:
project_id = "3w32"
pdb_id = project_id
cache_dir = Path(f".tmp/{project_id}")
cache_dir.mkdir(exist_ok=True, parents=True)
pdb_filepath_raw = cache_dir / "receptor_raw.pdb"
pdb_filepath_final = cache_dir / "receptor_fixed_apo.pdb"
dogsite_pockets_filepath = cache_dir / "pockets.pkl"
detector_filepath = cache_dir / "detector.pkl"

In [3]:
if not pdb_filepath_raw.is_file():
    pdb_file_content = sciapi.pdb.file.entry(pdb_id=pdb_id, file_format="pdb")
    pdb_filepath_raw.write_bytes(pdb_file_content)

In [4]:
if not pdb_filepath_final.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_raw))
    fixer.removeHeterogens(keepWater=False)
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_final, 'w'))

In [5]:
receptor = caddpy.chemsys.from_pdb(pdb_filepath_final)

In [9]:
if not detector_filepath.is_file():
    t_start=time.time()
    detector = caddpy.pocket.grid_detector.from_chemsys(receptor, gui=True, display=False, grid=0.4)
    t_end=time.time()
    print("Calculation time:", t_end - t_start)
    with open(detector_filepath, "wb") as f:
        pickle.dump(detector._detector, f)
else:
    with open(detector_filepath, "rb") as f:
        detector = caddpy.pocket.grid_detector.GridDetectorGUI(detector=pickle.load(f))

Add binding pockets detected by DoGSiteScorer (via ProteinsPlus web-API) for comparison:

In [10]:
if not dogsite_pockets_filepath.is_file():
    protplus_upload_results = sciapi.proteinsplus().upload_pdb(detector.receptor.to_pdb().to_file().encode())
    dogsite_results = sciapi.proteinsplus().dogsite(
        pdb_id=protplus_upload_results.dummy_pdb_id,
        algorithm="scorer",
    )
    dogsites = dogsite_results.full_data
    with open(dogsite_pockets_filepath, "wb") as f:
        pickle.dump(dogsites, f)
else:
    with open(dogsite_pockets_filepath, "rb") as f:
        dogsites = pickle.load(f)

for dogsite in dogsites:
    dogsite_pocket = scifile.mrc.read(dogsite.pop("mrc"))
    detector.nglwidget.add_volume(
        dogsite_pocket.data,
        basis=dogsite_pocket.grid_vectors,
        origin=dogsite_pocket.grid_origin,
        name=f"DoG{dogsite["name"].removeprefix("P")}",
        representation_params=scishow.nglview.SurfaceRepresentationParameters(
            lazy=True, opacity=0.8, contour=True, visible=True, color=(30,30,30), isolevel=1, isolevel_type="value"
        )
    )

Display GUI:

In [12]:
detector.display()

NGLWidget(gui_style='ngl')

Accordion(children=(Output(),), titles=('Logs',))